# Wavelength calibration with the maintained Python API

Install `python -m pip install -e "./python[fits,notebook]"` from the repository root. Select that Python kernel. This notebook uses the corrected ordered DTW + RANSAC implementation. Fitted-pair RMS is not independent laboratory accuracy.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from ransac_wv_calib import Parameters, calibrate, load_spectrum

# Find the repository when launched from its root or this examples directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples/data/lamp_calib").exists())
target_path = ROOT / "examples/data/Lamp_3s_19Aur_360_5s_1D_3.txt"
reference_path = ROOT / "examples/data/lamp_calib"
target = load_spectrum(target_path)
reference = load_spectrum(reference_path)
parameters = Parameters()


In [ ]:
result = calibrate(target, reference, parameters)
result["summary"]


In [ ]:
def norm(y):
    return (y-y.min()) / np.ptp(y)

tx = norm(target[:, 0])
rx = norm(reference[:, 0])
ty, ry = norm(target[:, 1]), norm(reference[:, 1])
calibrated = np.array(result["calibrated_spectrum"])
fig, axes = plt.subplots(3, 1, figsize=(14, 11), constrained_layout=True)
axes[0].plot(tx, ty, label=target_path.name)
axes[0].plot(rx, ry, label=reference_path.name)
for pair in result["matches"]:
    ti, ri = pair["target_index"], pair["reference_index"]
    axes[0].plot([tx[ti], rx[ri]], [ty[ti], ry[ri]], color="green" if pair["inlier"] else "gray", alpha=.6)
axes[0].set(xlabel="Normalized coordinate", ylabel="Normalized intensity", title="Ordered peak pairs (green: inlier; gray: outlier)")
axes[1].plot(calibrated[:, 1], ty, label="Calibrated target")
axes[1].plot(reference[:, 0], ry, label="Observed reference")
axes[1].set(xlabel="Wavelength, Å", ylabel="Normalized intensity", title="Review alignment, especially outside the outermost accepted anchors")
for accepted, color in [(True, "green"), (False, "gray")]:
    pairs = [p for p in result["matches"] if p["inlier"] == accepted]
    axes[2].scatter([p["reference_wavelength"] for p in pairs], [p["residual"] for p in pairs], color=color, label="Inliers" if accepted else "Outliers")
axes[2].axhline(0, color="black", linewidth=.7)
axes[2].set(xlabel="Reference wavelength, Å", ylabel="Reference − fit, Å", title=f"Fitted-pair RMS: {result["summary"]["rms_angstrom"]:.3f} Å")
for ax in axes:
    ax.grid(alpha=.25)
    ax.legend(loc="upper left", fontsize=8)
plt.show()


To use another spectrum, change both paths above as needed and rerun. The reference must have known wavelengths, the same approximate range and direction, and consistent units and air/vacuum convention. This notebook does not overwrite input data or save outputs automatically.
